# ENA reads from the Qiita dev stack

Read the per-run parquet with DuckDB + miint, run miint read QC, stream reads into deacon (host depletion) and sylph (profiling), and load the profiles.

Kernel: **qiita-reads** (`$QDEV_CONDA`). Data: `$QDEV_SHARE`.

In [ ]:
import os, subprocess, json
from pathlib import Path
import duckdb, pandas as pd, matplotlib.pyplot as plt

SHARE = Path(os.environ.get("QDEV_SHARE", f"/ddn_scratch/{os.environ['USER']}/qiita-dev-share"))
PIPE = SHARE / "pipeline"
con = duckdb.connect(config={"allow_unsigned_extensions": "true"})
con.execute("INSTALL miint FROM 'https://ftp.microbio.me/pub/miint'; LOAD miint")
con.sql("SELECT miint_version()").fetchone()

## Manifest: one row per run, with its Qiita study

In [ ]:
con.execute(f"CREATE OR REPLACE VIEW runs AS SELECT * FROM read_csv('{SHARE}/manifest.tsv', delim='\t', header=true)")
con.sql("""
  SELECT qiita_study_idx, study_accession, any_value(study_title) AS title, prep_protocol, platform, layout,
         count(*) AS runs, sum(read_count) AS reads, round(sum(parquet_bytes) / 1e9, 2) AS gb
  FROM runs GROUP BY ALL ORDER BY qiita_study_idx, prep_protocol
""").df()

## Read one run

Each parquet has `sequence_idx, read_id, sequence1, qual1, sequence2, qual2`; `*2` is NULL for single-end. `qual*` is a list of Phred scores.

In [ ]:
RUN = "SRR7947172"  # any run_accession from the manifest
path = con.sql(f"SELECT path FROM runs WHERE run_accession = '{RUN}'").fetchone()[0]
con.execute(f"CREATE OR REPLACE VIEW r AS SELECT * FROM read_parquet('{path}')")
con.sql("SELECT read_id, length(sequence1) AS len1, length(sequence2) AS len2 FROM r LIMIT 5").df()

In [ ]:
# pandas, if you prefer (pulls the whole run into memory, so pick columns):
df = con.sql("SELECT read_id, sequence1, sequence2 FROM r LIMIT 100000").df()
df.head()

## miint read QC

`filter_read(seq, qual)` returns pass/fail plus length, N count and mean quality per read (fastp defaults).

In [ ]:
qc = con.sql("""
  WITH f AS (SELECT filter_read(sequence1, qual1) AS q FROM r)
  SELECT q.passed, q.fail_reason, count(*) AS reads, round(avg(q.mean_quality), 1) AS mean_q, round(avg(q.length)) AS mean_len
  FROM f GROUP BY ALL ORDER BY reads DESC
""").df()
qc

In [ ]:
lens = con.sql("SELECT length(sequence1) AS len, count(*) AS n FROM r GROUP BY 1 ORDER BY 1").df()
plt.bar(lens.len, lens.n, width=1); plt.xlabel("R1 length (bp)"); plt.ylabel("reads"); plt.title(RUN); plt.show()

## Write FASTQ (optional)

miint's writer; `{ORIENTATION}` gives `_R1`/`_R2` for paired runs. Output is BGZF (standard gzip).

In [ ]:
# con.execute(f"""COPY (SELECT read_id, sequence1, qual1, sequence2, qual2 FROM r ORDER BY sequence_idx)
#                TO '/ddn_scratch/<you>/{RUN}_{{ORIENTATION}}.fastq.gz' (FORMAT FASTQ)""")

## Host depletion + profiling (metagenomic runs only)

Per run: parquet → miint FASTQ on stdout → `deacon filter -d` (panhuman-1) → depleted FASTQ → `sylph sketch`. Then one `sylph profile` over all sketches against GTDB r232, and `sylph-tax` for taxonomy.

For many runs use the Slurm scripts from a terminal:

```
$PIPE/submit_runs.sh /ddn_scratch/<you>/profiling/<name> PRJNA493153 PRJNA1083602   # array, one task per run
$PIPE/profile.sh     /ddn_scratch/<you>/profiling/<name>                             # after the array finishes
```

The cell below runs one run in-process (small runs only).

In [ ]:
OUT = Path(f"/ddn_scratch/{os.environ['USER']}/profiling/notebook")  # change to your own directory
# subprocess.run(["bash", str(PIPE / "deplete_sketch.sh"), RUN, str(OUT), "8"], check=True)
# json.load(open(OUT / "PRJNA493153" / RUN / "deacon.json"))

## Load profiles

`profile.tsv` (sylph): per sample × genome, with `Taxonomic_abundance`, `Sequence_abundance`, `Adjusted_ANI`. `taxprof/*.sylphmpa`: MetaPhlAn-style per-sample taxonomy.

In [ ]:
PROF = Path(f"/ddn_scratch/{os.environ['USER']}/profiling/test-PRJNA493153")
prof = pd.read_csv(PROF / "profile.tsv", sep="\t")
prof["run_accession"] = prof["Sample_file"].map(lambda p: Path(p).name.split(".")[0].split("_")[0])
prof = prof.merge(con.sql("SELECT run_accession, qiita_study_idx, study_accession, platform FROM runs").df(), on="run_accession")
prof.groupby(["run_accession", "platform"]).size().rename("species_detected").reset_index().head()

In [ ]:
def taxprof(path):
    """Species rows of a sylph-tax .sylphmpa file (MetaPhlAn-style; line 1 is '#SampleID ...')."""
    t = pd.read_csv(path, sep="\t", skiprows=1)
    return t[t.clade_name.str.contains("s__") & ~t.clade_name.str.contains("t__")]

files = sorted((PROF / "taxprof").glob("*.sylphmpa"))
top = taxprof(files[0]).nlargest(15, "relative_abundance")
top.assign(species=top.clade_name.str.split("s__").str[-1])[["species", "relative_abundance", "sequence_abundance"]]